# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`

This notebook demonstrates step-by-step how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` fields in accordance with Croissant standards.

### Dataset source

The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview

Review the available record sets, fields, and their IDs. All entities are referenced by their `@id`.

In [ ]:
# Identify record sets defined in the Croissant schema
record_sets = list(dataset.record_sets.keys())
print("Record sets (@id):")
for rs_id in record_sets:
    record_set = dataset.record_sets[rs_id]
    print(f"- {rs_id}: {getattr(record_set, 'name', '')}")
    # List all fields
    if hasattr(record_set, 'fields'):
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - {field['@id']} ({field.get('name', '')})")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis.

_Replace the example `record_sets_ids` below with the actual list from the previous cell; using only the record sets actually found in this dataset._

In [ ]:
# Use the discovered record set IDs from above
dataframes = dict()

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded {len(df)} records for record set: {record_set_id}")
        print("Columns:", df.columns.tolist())
        display(df.head())
    else:
        print(f"\nNo records found for record set: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering, normalizing numeric fields, and grouping data by attributes. All field references use their `@id`s as column labels.

In [ ]:
# For demonstration, pick the first non-empty record set and a likely numeric field id
if dataframes:
    # Get first available record set
    rs_id = next(iter(dataframes))
    df = dataframes[rs_id]
    print(f"Working with record set: {rs_id}")
    
    # Attempt to find a numeric field (float/int)
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    
    if numeric_field:
        print(f"Using numeric field (by @id): {numeric_field}")
        threshold = df[numeric_field].quantile(0.75) if len(df) > 0 else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records where {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Z-normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a non-numeric field for grouping
        group_field = None
        for col in df.columns:
            if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field:
            print(f"\nGrouping by {group_field} (by @id):")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found in the record set to analyze.")
else:
    print("No dataframes found to analyze.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. All axes and data references use their `@id` for traceability.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric distribution and, if present, grouped means
if 'numeric_field' in locals() and numeric_field is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("count")
    plt.show()

    if 'group_field' in locals() and group_field is not None:
        # Show group means as bar plot
        grouped_df = df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print("No suitable numeric field/EDA results available for visualization.")

## 6. Conclusion

- This notebook demonstrated how to load and interrogate a Croissant-compliant dataset via the `mlcroissant` API.
- All dataset elements were referenced by their `@id`, providing robust data lineage and traceability.
- Further domain-specific analysis can be built on top of these exploratory steps using the referenced record sets and field IDs provided by the Croissant metadata.

**For further analysis, consult the Croissant schema (`fair2.json`) or the [mlcroissant documentation](https://github.com/mlcommons/croissant).**